![Week 3 Day 4](../../assets/banners/w3d4.png)

# Week 3 Day 4 - Solution: Flagging Short Answers for Asha

The exercise: a second reducer key `flags`, a `review` node that interrupts **only** when something is flagged, mark overrides from Asha, and a resume from a brand-new graph object via `SqliteSaver`.

⏮ Back to the lab: `week3/day4.ipynb`

In [ ]:
import json
import operator
import sqlite3
from typing import Annotated, TypedDict
from uuid import uuid4

from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, Send, interrupt

from gradewise.config import DATA_DIR, OUTPUT_DIR
from gradewise.schemas import ExamPaper

exam = ExamPaper.model_validate_json(
    (DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json").read_text(encoding="utf-8")
)
Q3 = next(q for q in exam.questions if q.id == "Q3")
transcripts = json.loads((DATA_DIR / "answer-sheets" / "synthetic" / "transcripts.json").read_text(encoding="utf-8"))
ANSWERS = {s: transcripts[s]["Q3"] for s in list(transcripts)[:5]}

In [ ]:
def keyword_mark_q3(answer: str) -> float:
    text = answer.lower()
    definition_hits = [k for k in Q3.criteria[0].keywords if k in text]
    conditions = [c for c in Q3.criteria[1].keywords if c in text]
    return (2.0 if len(definition_hits) >= 2 else 1.0 if definition_hits else 0.0) + min(len(conditions), 2)


class ClassState(TypedDict, total=False):
    answers: dict[str, str]
    results: Annotated[list[dict], operator.add]
    flags: Annotated[list[str], operator.add]  # 💡 a second reducer key: parallel tasks append to it too
    overrides: dict[str, float]
    summary: str


def fan_out(state: ClassState) -> list[Send]:
    return [Send("mark_answer", {"student": s, "answer": a}) for s, a in state["answers"].items()]


def mark_answer(task: dict) -> dict:
    update = {"results": [{"student": task["student"], "awarded": keyword_mark_q3(task["answer"])}]}
    words = len(task["answer"].split())
    if words < 30:
        update["flags"] = [f"{task['student']} ({words} words)"]
    return update


def review(state: ClassState) -> dict:
    if not state.get("flags"):
        return {"overrides": {}}  # 💡 nothing suspicious: no interrupt at all
    flagged = [f.split()[0] for f in state["flags"]]
    decision = interrupt({"flags": state["flags"], "answers": {s: state["answers"][s] for s in flagged}})
    return {"overrides": decision or {}}


def summary(state: ClassState) -> dict:
    marks = {r["student"]: r["awarded"] for r in state["results"]} | state.get("overrides", {})
    return {"summary": " · ".join(f"{s}: {m:g}" for s, m in sorted(marks.items()))}


def build(checkpointer):
    builder = StateGraph(ClassState)
    for name, fn in [("mark_answer", mark_answer), ("review", review), ("summary", summary)]:
        builder.add_node(name, fn)
    builder.add_conditional_edges(START, fan_out, ["mark_answer"])
    builder.add_edge("mark_answer", "review")
    builder.add_edge("review", "summary")
    builder.add_edge("summary", END)
    return builder.compile(checkpointer=checkpointer)


def saver() -> SqliteSaver:
    path = OUTPUT_DIR / "week3_day4_solution.db"
    path.parent.mkdir(parents=True, exist_ok=True)
    return SqliteSaver(sqlite3.connect(str(path), check_same_thread=False))

In [ ]:
config = {"configurable": {"thread_id": f"short-{uuid4().hex[:6]}"}}
paused = build(saver()).invoke({"answers": ANSWERS, "results": [], "flags": []}, config)
payload = paused["__interrupt__"][0].value
print("⏸ Flagged:", payload["flags"])
for student, answer in payload["answers"].items():
    print(f"   {student}: {answer!r}")

In [ ]:
# 🔄 A brand-new graph object and a brand-new connection: as if Python had restarted
fresh = build(saver())
print("Still waiting at:", fresh.get_state(config).next)
final = fresh.invoke(Command(resume={"student_04": 3.5}), config)  # 🧑‍🏫 Asha's override
print("📊", final["summary"])

In [ ]:
# ✅ And with no short answers, the graph never pauses
long_only = {s: a + " I revised this topic carefully before the exam, honestly." * 2 for s, a in ANSWERS.items()}
result = build(saver()).invoke(
    {"answers": long_only, "results": [], "flags": []}, {"configurable": {"thread_id": f"long-{uuid4().hex[:6]}"}}
)
print("Paused?", "__interrupt__" in result, "·", result["summary"])